In [1]:
import pandas as pd

Here we use the chembl_websource_client to scrape all of the data points related to human liver microsom clearance.

For this we define the keywords:
    "human liver microsom",
    "human liver microsomal",
    "hlm",

and add that they should be CL, meaning related to clearance

In [2]:
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.settings import Settings
import csv
import os

# Optional client tuning
Settings.Instance().TIMEOUT = 30
Settings.Instance().CACHING = True
Settings.Instance().MAX_LIMIT = 1000

assay = new_client.assay
activity = new_client.activity
molecule = new_client.molecule
target = new_client.target

OUTPUT_DIR = "raw_datasets/chembl"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def chunks(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]


def is_hlm_description(text: str) -> bool:
    if not text:
        return False
    t = text.lower()
    return (
        "human" in t and
        "liver" in t and
        "microsom" in t   # catches microsome / microsomal
    )


# 1) Find candidate HLM assays
candidate_terms = [
    "human liver microsom",
    "human liver microsomal",
    "hlm",
]

assays = {}

for term in candidate_terms:
    results = assay.filter(description__icontains=term).only(
        ["assay_chembl_id", "description", "document_chembl_id", "assay_type"]
    )
    for a in results:
        desc = a.get("description") or ""
        if is_hlm_description(desc):
            assays[a["assay_chembl_id"]] = {
                "description": desc,
                "document_chembl_id": a.get("document_chembl_id"),
                "assay_type": a.get("assay_type"),
            }

assay_ids = list(assays.keys())
print(f"Found {len(assay_ids)} candidate HLM assays")

# 2) Pull CL activities from those assays
rows = []

for batch in chunks(assay_ids, 100):
    acts = activity.filter(
        assay_chembl_id__in=batch,
        standard_type="CL"
    ).only([
        "activity_id",
        "assay_chembl_id",
        "molecule_chembl_id",
        "target_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "data_validity_comment",
        "document_chembl_id",
        "document_year",
        "activity_comment",
    ])

    for x in acts:
        val = x.get("standard_value")
        mol_id = x.get("molecule_chembl_id")
        if val in (None, "") or not mol_id:
            continue

        rows.append({
            "activity_id": x.get("activity_id"),
            "molecule_chembl_id": mol_id,
            "target_chembl_id": x.get("target_chembl_id"),
            "assay_chembl_id": x.get("assay_chembl_id"),
            "document_chembl_id": x.get("document_chembl_id"),
            "data_year": x.get("document_year"),
            "standard_type": x.get("standard_type"),
            "standard_relation": x.get("standard_relation"),
            "standard_value": x.get("standard_value"),
            "standard_units": x.get("standard_units"),
            "data_validity_comment": x.get("data_validity_comment"),
            "activity_comment": x.get("activity_comment"),
            "assay_description": assays[x["assay_chembl_id"]]["description"],
            "assay_type": assays[x["assay_chembl_id"]]["assay_type"],
        })

print(f"Collected {len(rows)} HLM CL rows")

# 3) Optional cleanup: keep exact values only
clean_rows = [
    r for r in rows
    if r["standard_relation"] in (None, "=")
]

print(f"Exact-value rows: {len(clean_rows)}")

# 4) Fetch SMILES for all molecules
requested_molecule_ids = sorted({r["molecule_chembl_id"] for r in clean_rows})
smiles_by_mol = {}
returned_molecule_ids = set()

for batch in chunks(requested_molecule_ids, 100):
    mols = molecule.filter(
        molecule_chembl_id__in=batch
    ).only([
        "molecule_chembl_id",
        "molecule_structures",
    ])

    for m in mols:
        mol_id = m.get("molecule_chembl_id")
        returned_molecule_ids.add(mol_id)

        mol_struct = m.get("molecule_structures") or {}
        smiles_by_mol[mol_id] = mol_struct.get("canonical_smiles")

print(f"Unique molecule IDs requested: {len(requested_molecule_ids)}")
print(f"Molecule records returned: {len(returned_molecule_ids)}")

missing_molecule_ids = set(requested_molecule_ids) - returned_molecule_ids
molecule_ids_with_blank_smiles = {
    mol_id for mol_id in returned_molecule_ids
    if not smiles_by_mol.get(mol_id)
}

print(f"Molecule IDs not returned at all: {len(missing_molecule_ids)}")
print(f"Molecule IDs returned but with blank SMILES: {len(molecule_ids_with_blank_smiles)}")

# 5) Fetch target names
target_ids = sorted({
    r["target_chembl_id"]
    for r in clean_rows
    if r.get("target_chembl_id")
})

targets_used = {}

for batch in chunks(target_ids, 100):
    target_rows = target.filter(
        target_chembl_id__in=batch
    ).only([
        "target_chembl_id",
        "pref_name",
    ])

    for t in target_rows:
        targets_used[t["target_chembl_id"]] = t.get("pref_name")

print(f"Unique targets used: {len(targets_used)}")

# 6) Add SMILES + target name + structure status to rows
for r in clean_rows:
    mol_id = r["molecule_chembl_id"]
    target_id = r.get("target_chembl_id")

    r["smiles"] = smiles_by_mol.get(mol_id)
    r["target_name"] = targets_used.get(target_id)

    if mol_id in missing_molecule_ids:
        r["structure_status"] = "molecule_not_returned"
    elif mol_id in molecule_ids_with_blank_smiles:
        r["structure_status"] = "blank_smiles"
    else:
        r["structure_status"] = "smiles_found"

# 7) Split rows into with-structure and no-structure
rows_with_structure = [r for r in clean_rows if r["structure_status"] == "smiles_found"]
rows_without_structure = [r for r in clean_rows if r["structure_status"] != "smiles_found"]

print(f"Rows with structure: {len(rows_with_structure)}")
print(f"Rows without structure: {len(rows_without_structure)}")

# 8) Write targets txt file in Python dict format
targets_path = os.path.join(OUTPUT_DIR, "chembl_hlm_cl_targets.txt")
with open(targets_path, "w", encoding="utf-8") as f:
    f.write("TARGETS = {\n")
    for target_id in sorted(targets_used):
        pref_name = targets_used[target_id]
        pref_name_escaped = (pref_name or "").replace("\\", "\\\\").replace('"', '\\"')
        f.write(f'    "{target_id}": "{pref_name_escaped}",\n')
    f.write("}\n")

print(f"Wrote {targets_path}")

# 9) Write CSVs
fieldnames = [
    "activity_id",
    "molecule_chembl_id",
    "smiles",
    "structure_status",
    "target_chembl_id",
    "target_name",
    "assay_chembl_id",
    "document_chembl_id",
    "data_year",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "data_validity_comment",
    "activity_comment",
    "assay_type",
    "assay_description",
]

csv_with_structure = os.path.join(OUTPUT_DIR, "chembl_hlm_cl.csv")
csv_without_structure = os.path.join(OUTPUT_DIR, "chembl_hlm_cl_no_structure.csv")

with open(csv_with_structure, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_with_structure)

with open(csv_without_structure, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_without_structure)

print(f"Wrote {csv_with_structure}")
print(f"Wrote {csv_without_structure}")

Found 14273 candidate HLM assays
Collected 19167 HLM CL rows
Exact-value rows: 15314
Unique molecule IDs requested: 14176
Molecule records returned: 14176
Molecule IDs not returned at all: 0
Molecule IDs returned but with blank SMILES: 0
Unique targets used: 22
Rows with structure: 15314
Rows without structure: 0
Wrote raw_datasets/chembl\chembl_hlm_cl_targets.txt
Wrote raw_datasets/chembl\chembl_hlm_cl.csv
Wrote raw_datasets/chembl\chembl_hlm_cl_no_structure.csv


There are multiple various assays related to HLM clearance, like intrinsic, hepatic etc. so we need to separate them, since we will work with intrinsic clearance

In [3]:
import csv
import os
import re
from collections import defaultdict

INPUT_CSV = "raw_datasets/chembl/chembl_hlm_cl.csv"
OUTPUT_DIR = "raw_datasets/chembl/"

# Ordered from most specific to least specific
CATEGORY_PATTERNS = [
    (
        "chembl_hlm_unbound",
        [
            r"\bunbound intrinsic\b",
            r"\bintrinsic unbound\b",
            r"\bclint\s*[,/_ -]?\s*u\b",
            r"\bclu\b",
            r"\bfu[,/_ -]?mic\b",
            r"\bmicrosomal unbound clearance\b",
            r"\bunbound clearance\b",
        ],
    ),
    (
        "chembl_hlm_intrinsic",
        [
            r"\bapparent intrinsic\b",
            r"\bintrinsic apparent\b",
            r"\bapparent clint\b",
            r"\bclint\s*[,/_ -]?\s*app\b",
            r"\bclintapp\b",
            r"\bclint[,/_ -]?app\b",
            r"\bintrinsic clearance\b",
            r"\bclint\b",
            r"\bcl int\b",
            r"\bmicrosomal intrinsic clearance\b",
        ],
    ),
    (
        "chembl_hlm_hepatic",
        [
            r"\bhepatic clearance\b",
            r"\bhepatic intrinsic clearance\b",
            r"\bscaled intrinsic\b",
            r"\bscaled clint\b",
            r"\bivive\b",
            r"\bin vivo clearance\b",
            r"\bml/min/kg\b",
            r"\bl/kg/h\b",
            r"\bper\s*kg\b",
        ],
    ),
]

NEGATIVE_PATTERNS = {
    "chembl_hlm_intrinsic": [
        r"\bunbound intrinsic\b",
        r"\bclint\s*[,/_ -]?\s*u\b",
        r"\bclu\b",
        r"\bfu[,/_ -]?mic\b",
        r"\bmicrosomal unbound clearance\b",
        r"\bhepatic clearance\b",
        r"\bhepatic intrinsic clearance\b",
        r"\bscaled intrinsic\b",
        r"\bscaled clint\b",
        r"\bivive\b",
        r"\bml/min/kg\b",
        r"\bl/kg/h\b",
        r"\bper\s*kg\b",
    ],
}

def normalize_text(*parts):
    text = " | ".join(str(p or "") for p in parts)
    text = text.lower()
    text = text.replace("appearant", "apparent")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def matches_any(text, patterns):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

def classify_row(row):
    text = normalize_text(
        row.get("assay_description"),
        row.get("activity_comment"),
        row.get("target_name"),
        row.get("standard_units"),
    )

    for category, patterns in CATEGORY_PATTERNS:
        if matches_any(text, patterns):
            negs = NEGATIVE_PATTERNS.get(category, [])
            if negs and matches_any(text, negs):
                continue
            return category, text

    # plain "apparent" is too ambiguous: keep in unknown
    if matches_any(text, [
        r"\bapparent clearance\b",
        r"\bapparent cl\b",
        r"\bapparent\b",
    ]):
        return "chembl_hlm_unknown", text

    return "chembl_hlm_unknown", text

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    with open(INPUT_CSV, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []

        extra_fields = ["classification_text", "classification_bucket"]
        out_fieldnames = fieldnames + [c for c in extra_fields if c not in fieldnames]

        buckets = defaultdict(list)

        for row in reader:
            bucket, text = classify_row(row)
            row["classification_bucket"] = bucket
            row["classification_text"] = text
            buckets[bucket].append(row)

    # Ensure all expected buckets exist, even if empty
    expected_buckets = [
        "chembl_hlm_intrinsic",
        "chembl_hlm_unbound",
        "chembl_hlm_hepatic",
        "chembl_hlm_unknown",
    ]
    for bucket in expected_buckets:
        buckets[bucket] = buckets.get(bucket, [])

    # Write one file per bucket
    for bucket in expected_buckets:
        rows = buckets[bucket]
        out_path = os.path.join(OUTPUT_DIR, f"{bucket}.csv")
        with open(out_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=out_fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f"Wrote {out_path} ({len(rows)} rows)")

    # Write a summary file
    summary_path = os.path.join(OUTPUT_DIR, "summary.txt")
    total = sum(len(v) for v in buckets.values())
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"Input file: {INPUT_CSV}\n")
        f.write(f"Total rows processed: {total}\n\n")
        for bucket in expected_buckets:
            f.write(f"{bucket}: {len(buckets[bucket])}\n")

    print(f"Wrote {summary_path}")

if __name__ == "__main__":
    main()

Wrote raw_datasets/chembl/chembl_hlm_intrinsic.csv (12137 rows)
Wrote raw_datasets/chembl/chembl_hlm_unbound.csv (438 rows)
Wrote raw_datasets/chembl/chembl_hlm_hepatic.csv (276 rows)
Wrote raw_datasets/chembl/chembl_hlm_unknown.csv (2463 rows)
Wrote raw_datasets/chembl/summary.txt
